- This script is will be used to collect the Petrinex CSV files, combine
- The files into a DataFrame and filter for the company and download the volumetrics for that operating company

In [14]:
# THIS SECTION OF THE CODE IS USED TO COMBINE ALL THE DATA DOWNLOADED
#FROM PETRINEX INTO ONE BIG DATAFRAME 
import importlib
import pandas as pd
import glob 
import os
import numpy as np
import json
from emissions_production_calc import *

In [15]:
#Variables 
petrinex_folder_path = '/Users/moadmin/Desktop/GitHub/Emissions-Quantification-Data-Pipeline/3_Emission_Quantification_Scripts/bq-results-20260216-203724-1771274431918.csv'

with open('quantification_config.json') as f:
    config = json.load(f)

emissions_factor_mapping = {
    'FUEL': {
        'CO2': config['emission_factors']['FUEL']['CO2']['value'],
        'CH4': config['emission_factors']['FUEL']['CH4']['value'],
        'N2O': config['emission_factors']['FUEL']['N2O']['value']
    },
    'FLARE': {
        'CO2': config['emission_factors']['FLARE']['CO2']['value'],
        'CH4': config['emission_factors']['FLARE']['CH4']['value'],
        'N2O': config['emission_factors']['FLARE']['N2O']['value']
    }
}

emissions_factor_unit_mapping = {
    'FUEL': {
        'CO2': config['emission_factors']['FUEL']['CO2']['unit'],
        'CH4': config['emission_factors']['FUEL']['CH4']['unit'],
        'N2O': config['emission_factors']['FUEL']['N2O']['unit']
    },
    'FLARE': {
        'CO2': config['emission_factors']['FLARE']['CO2']['unit'],
        'CH4': config['emission_factors']['FLARE']['CH4']['unit'],
        'N2O': config['emission_factors']['FLARE']['N2O']['unit']
    }
}

In [16]:
# Load the dataframe from the CSV
source_df = pd.read_csv(petrinex_folder_path)

# test update for for and from date 
source_df['effective_from_date'] = pd.to_datetime('2026-01-25')
source_df['effective_to_date'] = pd.to_datetime(None)

source_df.head()

,RecordDate,PetrinexFacilityID,ActivityId,Product,Volume,units,is_current,effective_from_date,effective_to_date
0,2022-05-01,ABBT0048421,DIFF,GAS,0.0,e3m3,True,2026-01-25,NaT
1,2022-05-01,ABBT0048421,DISP,GAS,57.5,e3m3,True,2026-01-25,NaT
2,2022-05-01,ABBT0048421,FUEL,GAS,0.1,e3m3,True,2026-01-25,NaT
3,2022-05-01,ABBT0048421,PROD,GAS,58.4,e3m3,True,2026-01-25,NaT
4,2022-05-01,ABBT0048421,VENT,GAS,0.8,e3m3,True,2026-01-25,NaT


In [17]:
# Load the dataframe from the CSV
source_df = pd.read_csv(petrinex_folder_path)

# Convert month ProductionMonth to datatime
source_df['RecordDate'] = pd.to_datetime(source_df['RecordDate'])

# Add the Year and Month columns
source_df['Year'] = source_df['RecordDate'].dt.year
source_df['Month'] = source_df['RecordDate'].dt.strftime('%b')

# Filter on the ActivityID to activities that resulting in emissions released
source_df = source_df[
    (source_df['ActivityId'] == 'FUEL') | 
    (source_df['ActivityId'] == 'FLARE') | 
    (source_df['is_current'] == 'True')
    ]

In [18]:
# Create the Emission factor columns for CO2, CH4, and N2O base on the ActivityId
source_df['CO2_Emission_Factor'] = np.select(
    [source_df['ActivityId'] == 'FUEL', source_df['ActivityId'] == 'FLARE'], # If condition for fuel and flare
    [emissions_factor_mapping['FUEL']['CO2'], emissions_factor_mapping['FLARE']['CO2']], # Results if condition is true for fuel or flare
    default=0.0
)

source_df['CH4_Emission_Factor'] = np.select(
    [source_df['ActivityId'] == 'FUEL', source_df['ActivityId'] == 'FLARE'], # If condition for fuel and flare
    [emissions_factor_mapping['FUEL']['CH4'], emissions_factor_mapping['FLARE']['CH4']], # Results if condition is true for fuel or flare
    default=0.0
)

source_df['N2O_Emission_Factor'] = np.select(
    [source_df['ActivityId'] == 'FUEL', source_df['ActivityId'] == 'FLARE'], # If condition for fuel and flare
    [emissions_factor_mapping['FUEL']['N2O'], emissions_factor_mapping['FLARE']['N2O']], # Results if condition is true for fuel or flare
    default=0.0
)

# Create the Emission factor Unit columns for CO2, CH4, and N2O base on the ActivityId
source_df['CO2_Emission_Factor_Unit'] = np.select(
    [source_df['ActivityId'] == 'FUEL', source_df['ActivityId'] == 'FLARE'], # If condition for fuel and flare
    [emissions_factor_unit_mapping['FUEL']['CO2'], emissions_factor_unit_mapping['FLARE']['CO2']], # Results if condition is true for fuel or flare
    default=None
)

source_df['CH4_Emission_Factor_Unit'] = np.select(
    [source_df['ActivityId'] == 'FUEL', source_df['ActivityId'] == 'FLARE'], # If condition for fuel and flare
    [emissions_factor_unit_mapping['FUEL']['CH4'], emissions_factor_unit_mapping['FLARE']['CH4']], # Results if condition is true for fuel or flare
    default=None
)

source_df['N2O_Emission_Factor_Unit'] = np.select(
    [source_df['ActivityId'] == 'FUEL', source_df['ActivityId'] == 'FLARE'], # If condition for fuel and flare
    [emissions_factor_unit_mapping['FUEL']['N2O'], emissions_factor_unit_mapping['FLARE']['N2O']], # Results if condition is true for fuel or flare
    default=None
)

In [23]:
#Add Columns for emission factor for flare and stationary fuel combustion
source_df[['CO2_Emissions_tonne'
        ,'CH4_Emissions_tonne'
        ,'N2O_Emissions_tonne'
        ,'Total_Emissions_TCO2e']] = ''


#filtered dataframe
filtered_source_df = source_df[[
    'RecordDate'
    ,'PetrinexFacilityID'
    ,'ActivityId'
    ,'Product'
    ,'Volume'
    ,'units'
    ,'CO2_Emission_Factor'
    ,'CO2_Emission_Factor_Unit'
    ,'CO2_Emissions_tonne'
    ,'CH4_Emission_Factor'
    ,'CH4_Emission_Factor_Unit'
    ,'CH4_Emissions_tonne'
    ,'N2O_Emission_Factor'
    ,'N2O_Emission_Factor_Unit'
    ,'N2O_Emissions_tonne'
    ,'Total_Emissions_TCO2e']]

In [24]:
#Create a new dataframe for emissions 
emissions_df_instance = ConOilnGasEmissionCalc(filtered_source_df)

emissions_final = emissions_df_instance.calculate_emissions(
                                                            fuel_type='ActivityId',
                                                            volume_col='Volume', 
                                                            co2_ef='CO2_Emission_Factor',
                                                            ch4_ef='CH4_Emission_Factor',
                                                            n2o_ef='N2O_Emission_Factor',
                                                            co2_result='CO2_Emissions_tonne',
                                                            ch4_result='CH4_Emissions_tonne',
                                                            n2o_result='N2O_Emissions_tonne',
                                                            total_emission='Total_Emissions_TCO2e'
                                                            )

emissions_final.head()

,RecordDate,PetrinexFacilityID,ActivityId,Product,Volume,units,CO2_Emission_Factor,CO2_Emission_Factor_Unit,CO2_Emissions_tonne,CH4_Emission_Factor,CH4_Emission_Factor_Unit,CH4_Emissions_tonne,N2O_Emission_Factor,N2O_Emission_Factor_Unit,N2O_Emissions_tonne,Total_Emissions_TCO2e
2,2022-05-01,ABBT0048421,FUEL,GAS,0.1,e3m3,0.00233,tonnes/m3,0.233,0.000006,tonnes/m3,0.000640,6.000000e-08,tonnes/m3,0.000006,0.252510
22,2022-05-01,ABBT0101896,FLARE,GAS,4.3,e3m3,2280.00000,g/m3,9.804,10.830000,g/m3,0.046569,3.300000e-02,g/m3,0.000142,11.145535
23,2022-05-01,ABBT0101896,FUEL,GAS,18.1,e3m3,0.00233,tonnes/m3,42.173,0.000006,tonnes/m3,0.115840,6.000000e-08,tonnes/m3,0.001086,45.704310
40,2022-05-01,ABBT0120813,FUEL,GAS,2.0,e3m3,0.00233,tonnes/m3,4.660,0.000006,tonnes/m3,0.012800,6.000000e-08,tonnes/m3,0.000120,5.050200
74,2022-05-01,ABBT2240034,FLARE,GAS,38.5,e3m3,2280.00000,g/m3,87.780,10.830000,g/m3,0.416955,3.300000e-02,g/m3,0.001270,99.791422


In [ ]:
test_df.head()

In [ ]:
# Specify the folder path consisting the CSV files
folder_path = '/Users/moadmin/Desktop/Programming Projects/Petrinex Analysis/Petrinex Source Data'
#list out all the files in the folder
files = os.listdir(folder_path)
# files # Display the file names

In [ ]:
# Initiallize an empty DataFrame to store the combined data
combined_data = pd.DataFrame()

# Look through the files in the folder 
for file_name in files: 
    file_path = os.path.join(folder_path, file_name)

    #Check if the file is a CSV file
    if file_name.endswith('.CSV') and os.path.isfile(file_path):
        try:
            #read the CSV file into a DataFrame
            df = pd.read_csv(file_path)

            # Append the dataframe to the combined_data DataFrame
            combined_data = combined_data.append(df, ignore_index = True)
            print(f"Read data from '{file_name} and appended to the Combined Data DataFrame")   # This line of code is not needed
        except Exception as e:
            print(f"Error reading '{file_name}: {e}")

# Print the combined DataFrame
# print("Combined Data:")
# print(combined_data)


In [ ]:
# Convert month ProductionMonth to datatime
combined_data['ProductionMonth'] = pd.to_datetime(combined_data['ProductionMonth'])

# Add the Year and Month columns
combined_data['Year'] = combined_data['ProductionMonth'].dt.year
combined_data['Month'] = combined_data['ProductionMonth'].dt.strftime('%b')

In [ ]:
# Filter on the ActivityID to activities that resulting in emissions released
emissions_df = combined_data[(combined_data['ActivityID'] == 'FUEL') | (combined_data['ActivityID'] == 'FLARE')]

#Add Columns for emission factor for flare and stationary fuel combustion
emissions_df[['CO2_Emission_Factor'
        ,'CO2_Emission_Factor_Unit'
        ,'CO2_Emissions_tonne'
        ,'CH4_Emission_Factor'
        ,'CH4_Emission_Factor_Unit'
        ,'CH4_Emissions_tonne'
        ,'N2O_Emission_Factor'
        ,'N2O_Emission_Factor_Unit'
        ,'N2O_Emissions_tonne'
        ,'Total_Emissions_TCO2e']] = ''


#filtered dataframe
filtered_emissions_df = emissions_df[['Year'
    ,'Month'
    ,'OperatorBAID'
    ,'OperatorName'
    ,'ReportingFacilityID'
    ,'ReportingFacilityName'
    ,'ReportingFacilityLocation'
    ,'ActivityID'
    ,'ProductID'
    ,'Volume'
    ,'CO2_Emission_Factor'
    ,'CO2_Emission_Factor_Unit'
    ,'CO2_Emissions_tonne'
    ,'CH4_Emission_Factor'
    ,'CH4_Emission_Factor_Unit'
    ,'CH4_Emissions_tonne'
    ,'N2O_Emission_Factor'
    ,'N2O_Emission_Factor_Unit'
    ,'N2O_Emissions_tonne'
    ,'Total_Emissions_TCO2e']]

filtered_emissions_df.head()

This section filters the company 
while True: 
    if operator_name in combined_data['OperatorName'].values:
        print("The Operator You're Filtering For Is: " + operator_name)
        break
    else:
        print("We did not find you operator name in our list. Please enter the Operator Name as it Appears in Petrinex, Your Entry is:" + operator_name)

In [ ]:
filtered_company = combined_data[combined_data['OperatorName']== operator_name]
filtered_company.head(5)

#THIS SECTION FILTERS MULTIPLE CRITERIA

filtered_company_list = ['JAPAN CANADA OIL SANDS LIMITED','GREENFIRE HANGINGSTONE OPERATING CORPORATION','GREENFIRE ACQUISITION CORPORATION','GREENFIRE RESOURCES OPERATING CORPORATION']
filtered_list = combined_data[combined_data.OperatorName.isin(filtered_company_list)]
filtered_list.head(100)